# Home Loan Model Monitoring
End-to-end PSI, CSI, KS, AUC

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression

In [ ]:
# Sample synthetic data
np.random.seed(42)
train = pd.DataFrame({
 'income': np.random.normal(50000,10000,1000),
 'age': np.random.randint(21,60,1000),
 'loan_amount': np.random.normal(200000,50000,1000),
 'cibil_score': np.random.randint(600,800,1000),
 'target': np.random.randint(0,2,1000)
})

prod = train.copy()
prod['income'] = prod['income']*1.1

In [ ]:
# Model scoring
features = ['income','age','loan_amount','cibil_score']
model = LogisticRegression()
model.fit(train[features], train['target'])

train['score'] = model.predict_proba(train[features])[:,1]
prod['score'] = model.predict_proba(prod[features])[:,1]

In [ ]:
# PSI

def psi(expected, actual, buckets=10):
    breakpoints = np.linspace(0,1,buckets+1)
    e = np.histogram(expected, breakpoints)[0]/len(expected)
    a = np.histogram(actual, breakpoints)[0]/len(actual)
    return np.sum((e-a)*np.log((e+1e-6)/(a+1e-6)))

psi_val = psi(train['score'], prod['score'])
psi_val

In [ ]:
# KS
ks, _ = ks_2samp(train['score'], prod['score'])
ks

In [ ]:
# AUC
auc = roc_auc_score(prod['target'], prod['score'])
gini = 2*auc -1
auc, gini

In [ ]:
# CSI

def csi(train, prod, col):
    bins = np.percentile(train[col], np.arange(0,101,10))
    t = pd.cut(train[col], bins).value_counts(normalize=True)
    p = pd.cut(prod[col], bins).value_counts(normalize=True)
    return np.sum((t-p)*np.log((t+1e-6)/(p+1e-6)))

for col in ['income','age','loan_amount','cibil_score']:
    print(col, csi(train, prod, col))